# Protein Pretrain — Windows Local

Windows version of `protein_pretrain.ipynb`.  
Assumes `train_part_*.txt` files are **already available locally** (no MinIO download needed).

## Prerequisites

1. Python 3.11+ installed
2. PyTorch with CUDA: `pip install torch --index-url https://download.pytorch.org/whl/cu121`
3. Install project: `pip install -e .` from repo root
4. Place `train_part_*.txt` files in `data/cache/protein_train_parts/`
5. (Optional) Set `HF_TOKEN` env var for HuggingFace upload

Supports:
- **train_from_scratch**: Build tokenizer, model, and train from epoch 0.
- **resume**: Restore from a checkpoint and continue training.
- **auto**: Resume if checkpoint exists, otherwise train from scratch.

In [1]:
import os
import sys
import types
import site
from pathlib import Path

REPO_DIR = Path(r"C:\Users\Admin\MDNAC").resolve()

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.environ.update({
    "MDNAC_STORAGE_BACKEND": "local",
    "USE_MINIO": "false",
    "MINIO_ENABLED": "false",
    "TOKENIZERS_PARALLELISM": "false",
    "KMP_DUPLICATE_LIB_OK": "TRUE",
    "PYTORCH_NVML_BASED_CUDA_CHECK": "1",
})

for sp in site.getsitepackages():
    torch_lib = Path(sp) / "torch" / "lib"
    if torch_lib.exists():
        os.environ["PATH"] = str(torch_lib) + os.pathsep + os.environ.get("PATH", "")
        try:
            os.add_dll_directory(str(torch_lib))
        except Exception:
            pass

import torch

print("✅ torch import OK")
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

# Bypass libs.core.__init__.py side effect
libs_pkg = types.ModuleType("libs")
libs_pkg.__path__ = [str(REPO_DIR / "libs")]
sys.modules["libs"] = libs_pkg

core_pkg = types.ModuleType("libs.core")
core_pkg.__path__ = [str(REPO_DIR / "libs" / "core")]
sys.modules["libs.core"] = core_pkg

pretrain_pkg = types.ModuleType("libs.core.pretrain")
pretrain_pkg.__path__ = [str(REPO_DIR / "libs" / "core" / "pretrain")]
sys.modules["libs.core.pretrain"] = pretrain_pkg

from libs.core.pretrain.training_config import load_protein_training_config
from libs.core.pretrain.protein_lm.trainer import ProteinPretrainTrainer
from libs.core.pretrain.protein_lm.core import discover_protein_train_text_paths


class WindowsLocalProteinPretrainTrainer(ProteinPretrainTrainer):
    """Notebook-only fallback for local Windows part files."""

    def _discover_local_paths(self):
        local_paths = super()._discover_local_paths()
        if local_paths:
            return local_paths

        cache_dir = Path(self._paths["train_part_cache_dir"])
        fallback_train_text_path = cache_dir / Path(self._paths["train_text_path"]).name
        try:
            return discover_protein_train_text_paths(
                fallback_train_text_path,
                part_glob=self._data_cfg["train_part_glob"],
                prefer_parts=True,
            )
        except FileNotFoundError:
            return ()

print("✅ Setup complete Windows local GPU — no MinIO")

✅ torch import OK
torch: 2.12.0+cu130
cuda: 13.0
cuda available: True
gpu: NVIDIA GeForce RTX 5060 Ti
✅ Setup complete Windows local GPU — no MinIO


In [2]:
CONFIG_PATH = REPO_DIR / "train.windows.yaml"

config = load_protein_training_config(REPO_DIR, config_path=CONFIG_PATH)

config_summary = {
    "mode": config["mode"],
    "optimizer_type": config["optimizer"]["type"],
    "device": config["runtime"]["device"],
    "multi_gpu_mode": config["runtime"]["multi_gpu_mode"],
    "mixed_precision": config["runtime"]["mixed_precision"],
    "checkpoint_dir": str(config["paths"]["checkpoint_dir"]),
    "resume_state_path": str(config["paths"]["resume_state_path"]),
    "minio_prefix": config["minio"]["train_parts_prefix_uri"] or None,
    "minio_uris_count": len(config["minio"]["train_part_uris"]),
    "train_part_cache_dir": str(config["paths"]["train_part_cache_dir"]),
    "num_epochs": config["training"]["num_epochs"],
    "max_steps": config["training"].get("max_steps"),
    "batch_size": config["data"]["batch_size"],
    "context_length": config["model"]["context_length"],
}
config_summary

{'mode': {'name': 'train_from_scratch', 'resume_if_available': True},
 'optimizer_type': 'muon',
 'device': 'auto',
 'multi_gpu_mode': 'auto',
 'mixed_precision': 'fp16',
 'checkpoint_dir': 'C:\\Users\\Admin\\MDNAC\\data\\checkpoints\\protein_from_scratch',
 'resume_state_path': 'C:\\Users\\Admin\\MDNAC\\data\\checkpoints\\protein_from_scratch\\resume_state.json',
 'minio_prefix': None,
 'minio_uris_count': 0,
 'train_part_cache_dir': 'C:\\Users\\Admin\\MDNAC\\data\\cache\\protein_train_parts',
 'num_epochs': 3,
 'max_steps': None,
 'batch_size': 2,
 'context_length': 512}

## Verify Local Data

Check that `train_part_*.txt` files exist in the cache directory.

In [3]:
# ── Verify local train parts ──────────────────────────────────────────────────
cache_dir = Path(config["paths"]["train_part_cache_dir"])
parts = sorted(cache_dir.glob("train_part_*.txt"))

if parts:
    total_size_mb = sum(p.stat().st_size for p in parts) / (1024 * 1024)
    print(f"✅ Found {len(parts)} train_part files ({total_size_mb:.1f} MB total)")
    print(f"   First: {parts[0].name}")
    print(f"   Last:  {parts[-1].name}")

    trainer_probe = WindowsLocalProteinPretrainTrainer(REPO_DIR, config_path=CONFIG_PATH)
    discovered_parts = trainer_probe._discover_local_paths()
    if not discovered_parts:
        raise FileNotFoundError("Trainer could not discover local train_part files.")
    print(f"Trainer will read {len(discovered_parts)} local part file(s)")
    print(f"   Source dir: {discovered_parts[0].parent}")
else:
    # Fallback: check for single train.txt
    train_txt = Path(config["paths"]["train_text_path"])
    if train_txt.exists():
        size_mb = train_txt.stat().st_size / (1024 * 1024)
        print(f"✅ Found single train.txt ({size_mb:.1f} MB)")
    else:
        print(f"❌ No training data found!")
        print(f"   Expected train_part_*.txt in: {cache_dir}")
        print(f"   Or single file at: {train_txt}")
        raise FileNotFoundError("No local training data. Place train_part_*.txt in the cache dir.")

✅ Found 5 train_part files (9376.3 MB total)
   First: train_part_1.txt
   Last:  train_part_5.txt
Trainer will read 5 local part file(s)
   Source dir: C:\Users\Admin\MDNAC\data\cache\protein_train_parts


## VRAM Check (16GB)

Before training, verify that the config fits within your GPU's VRAM budget.

In [4]:
# ── VRAM Preflight Check ──────────────────────────────────────────────────────
from libs.data.training.tokenizer import SequenceTokenizer
from libs.core.pretrain.protein_lm.memory import (
    estimate_protein_pretrain_memory,
    recommend_16gb_train_config,
    _resolve_dtype_from_mixed_precision,
)
from libs.core.pretrain.protein_lm.support.backbone import (
    build_mdc_config_from_progen_config,
    build_progen_config,
)

# Load real tokenizer
tokenizer_map_path = config["paths"]["tokenizer_map_path"]
assert tokenizer_map_path.exists(), f"tokenizer_map.json not found: {tokenizer_map_path}"
tokenizer = SequenceTokenizer.load_map(tokenizer_map_path)
print(f"✅ Tokenizer loaded — vocab_size = {tokenizer.vocab_size}")

# Build model config with mixed precision dtype
mixed_precision = config["runtime"]["mixed_precision"]
resolved_dtype = _resolve_dtype_from_mixed_precision(mixed_precision)
model_cfg = config["model"]
progen_config = build_progen_config(
    model_cfg["progen_model_size"],
    vocab_size=tokenizer.vocab_size,
    context_length=model_cfg["context_length"],
    dtype=resolved_dtype,
)
overrides = model_cfg["progen_config_overrides"]
if overrides:
    progen_config = {**progen_config, **overrides}
model_config = build_mdc_config_from_progen_config(progen_config, dtype=resolved_dtype)

# Estimate VRAM
import torch
estimate = estimate_protein_pretrain_memory(
    model_config=model_config,
    tokenizer=tokenizer,
    batch_size=config["data"]["batch_size"],
    context_length=model_cfg["context_length"],
    optimizer_type=config["optimizer"]["type"],
    dtype=resolved_dtype,
    mixed_precision=mixed_precision,
)

MAX_VRAM_GB = 16.0
TARGET_FRACTION = 0.85
target_budget = min(MAX_VRAM_GB * TARGET_FRACTION, MAX_VRAM_GB - 2.0)

print(f"\n{'='*60}")
print(f"VRAM ESTIMATE (resolved_vocab_size={tokenizer.vocab_size})")
print(f"{'='*60}")
print(f"  Parameters:      {estimate['param_count']:,} ({estimate['param_memory_gb']:.3f} GB)")
print(f"  Gradients:       {estimate['gradient_memory_gb']:.3f} GB")
print(f"  Optimizer state: {estimate['optimizer_state_gb']:.3f} GB")
print(f"  Activations:     {estimate['activation_memory_gb']:.3f} GB")
print(f"  Logits:          {estimate['logits_memory_gb']:.3f} GB")
print(f"  TOTAL:           {estimate['total_estimate_gb']:.3f} GB")
print(f"  Budget:          {target_budget:.3f} GB")
print(f"  Margin:          {target_budget - estimate['total_estimate_gb']:+.3f} GB")

fits = estimate["total_estimate_gb"] <= target_budget
if fits:
    print(f"\n  ✓ Estimated to fit within {MAX_VRAM_GB:.0f}GB")
else:
    print(f"\n  ✗ EXCEEDS {MAX_VRAM_GB:.0f}GB budget!")

if not torch.cuda.is_available():
    print("  ⚠ No CUDA — this is only an estimate. Actual peak may be higher.")

✅ Tokenizer loaded — vocab_size = 256
The MDC fast path is unavailable (missing optional libraries: causal-conv1d, flash-linear-attention). Falling back to the torch implementation. The fallback uses fp32 computation (2x VRAM for activations).

VRAM ESTIMATE (resolved_vocab_size=256)
  Parameters:      280,021,632 (0.522 GB)
  Gradients:       1.043 GB
  Optimizer state: 1.045 GB
  Activations:     0.438 GB
  Logits:          0.000 GB
  TOTAL:           3.048 GB
  Budget:          13.600 GB
  Margin:          +10.552 GB

  ✓ Estimated to fit within 16GB


In [5]:
# ── Train ─────────────────────────────────────────────────────────────────────
trainer = WindowsLocalProteinPretrainTrainer(REPO_DIR, config_path=CONFIG_PATH)
result = trainer.train()
result

🚀 Training mode: train_from_scratch
📦 [1/6] Loading tokenizer...
✅ Tokenizer loaded — vocab_size=256
🧠 [2/6] Building model...


✅ Model built — 280,021,632 parameters
⚙️  [3/6] Preparing runtime...
✅ Device: cuda | Distributed: False
🔧 [4/6] Creating optimizer...
✅ Optimizer ready
🔍 Running VRAM preflight check (target=14.0GB)...
✅ Preflight passed — peak=5.07GB / target=14.0GB
📂 [5/6] Building data loaders...
✅ Data loaders ready
🏋️ [6/6] Starting training loop...
📈 Epoch 1/3 | step=0 | tokens=0
  🔄 step=50 | loss=4.9664 | tokens=152,222
  🔄 step=100 | loss=4.9706 | tokens=303,349
  📊 eval step=100 | train=4.7545 | val=4.9336 🏆 new best val_loss!
  💾 Saving checkpoint at step 100...
  🔄 step=150 | loss=4.4878 | tokens=448,283
  🔄 step=200 | loss=4.8369 | tokens=600,725
  📊 eval step=200 | train=4.8076 | val=4.8711 🏆 new best val_loss!
  💾 Saving checkpoint at step 200...
  🔄 step=250 | loss=4.7339 | tokens=744,940
  🔄 step=300 | loss=4.4160 | tokens=887,144
  📊 eval step=300 | train=4.7579 | val=4.8833
  💾 Saving checkpoint at step 300...
  🔄 step=350 | loss=4.7925 | tokens=1,031,339
  🔄 step=400 | loss=4.5650

KeyboardInterrupt: 

In [ ]:
# ── Generate sample ───────────────────────────────────────────────────────────
from libs.core.pretrain.protein_lm.core import generate_protein_text

if trainer.is_main_process and trainer.model is not None:
    sample = generate_protein_text(
        trainer.model,
        trainer.tokenizer,
        prompt="<|protein|>",
        device=trainer.runtime.device,
        max_new_tokens=80,
        context_length=int(trainer.model_config.context_length),
    )
    print(sample)
else:
    print("Sample generation is emitted on rank 0 only.")

In [ ]:
# ── Upload model to HuggingFace (Optional) ────────────────────────────────────
from huggingface_hub import HfApi, login

HF_REPO_ID = "admincybers2/MDNAC-protein-pretrain"  # ← change repo name as needed

login(token=os.environ["HF_TOKEN"])
api = HfApi()

checkpoint_path = result.checkpoint_path
best_path = result.best_checkpoint_path

files_to_upload = []
if checkpoint_path and checkpoint_path.exists():
    files_to_upload.append((str(checkpoint_path), checkpoint_path.name))
if best_path and best_path.exists():
    files_to_upload.append((str(best_path), best_path.name))

# Upload tokenizer_map.json alongside
tokenizer_map = config["paths"]["tokenizer_map_path"]
if Path(tokenizer_map).exists():
    files_to_upload.append((str(tokenizer_map), "tokenizer_map.json"))

# Upload resume_state.json
resume_state = config["paths"]["resume_state_path"]
if Path(resume_state).exists():
    files_to_upload.append((str(resume_state), "resume_state.json"))

for local_path, repo_filename in files_to_upload:
    print(f"Uploading {repo_filename} ...")
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=repo_filename,
        repo_id=HF_REPO_ID,
        repo_type="model",
    )

print(f"✅ Uploaded {len(files_to_upload)} file(s) to https://huggingface.co/{HF_REPO_ID}")